# 04 · Transport evaluation
Leave-one-site-out and real covariate-shift splits inside RICH, then the guarded RICH ↔ NAGALA cross-cohort analysis, subgroup metrics, recalibration and paired bootstrap.

**Protocol rule:** the cross-cohort cells stay locked until `protocol.md` is locked and lambda is fixed.

In [ ]:
import os, sys, pathlib, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
sys.path.insert(0, ".")
import ckinn_lib as L

DATA = pathlib.Path(os.environ.get("CKINN_DATA", "data"))        # harmonized CSVs from notebook 01
OUT = pathlib.Path("results"); OUT.mkdir(exist_ok=True)
FIG = pathlib.Path("figs"); FIG.mkdir(exist_ok=True)
FAST = os.environ.get("CKINN_FAST", "0") == "1"                    # FAST=1: small subsample, code check only
SEED = 0
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
rich, nag = L.load_data(DATA)
if FAST:
    rich = rich.sample(40000, random_state=SEED)                   # development cohort only
print(f"RICH {len(rich):,} rows, {rich.event.sum():,} events | FAST={FAST}")

## 1. Configuration

In [ ]:
TIER_LIST = ["T2"] if FAST else ["T1", "T2", "T3"]
LAM = 1.0                          # set from notebook 03 (decision log)
EPOCHS = 6 if FAST else 40
XGB_N = 100 if FAST else 300
PROTOCOL_LOCKED = False            # set True only after protocol.md is locked and signed off
def specs(cols):
    return {"Cox (ridge)": lambda: L.CoxLinear(cols),
            "XGB-Cox monotone": lambda: L.XGBCox(cols, True, XGB_N, seed=SEED),
            "MLP": lambda: L.DeepCox(cols, lam=0.0, epochs=EPOCHS, seed=SEED),
            "CKINN": lambda: L.DeepCox(cols, lam=LAM, epochs=EPOCHS, seed=SEED)}

## 2. R2 · Leave-one-site-out within RICH

In [ ]:
rows = []
for tier in TIER_LIST:
    for i, (site, tr, te) in enumerate(L.loso_folds(rich)):
        if FAST and i >= 2: break
        if te.event.sum() < 20: continue
        res, _ = L.run_models(specs(L.TIERS[tier]), tr, te)
        res.insert(0, "tier", tier); res.insert(1, "site", site); rows.append(res)
        print(tier, "site", site, "done")
loso = pd.concat(rows, ignore_index=True); loso.to_csv(OUT / "04_loso.csv", index=False)
loso.groupby(["tier", "model"])[["cindex", "scaled_brier", "oe", "cal_slope", "viol_rate"]].agg(["mean", "std"]).round(3)

## 3. R4 · Real covariate shift inside RICH

In [ ]:
rows = []
for var, cut in [("age", 45), ("bmi", 25)]:
    tr, te = L.shift_split(rich, var, cut)
    for tier in TIER_LIST:
        res, _ = L.run_models(specs(L.TIERS[tier]), tr, te)
        res.insert(0, "shift", f"train {var}<{cut} -> test >={cut}"); res.insert(1, "tier", tier); rows.append(res)
shift = pd.concat(rows, ignore_index=True); shift.to_csv(OUT / "04_covariate_shift.csv", index=False)
shift[["shift", "tier", "model", "cindex", "scaled_brier", "oe", "cal_slope", "viol_rate"]].round(3)

## 4. Cross-cohort function (tested here on a RICH-only dry run)
`cross_cohort` trains on one cohort and tests on another using the shared columns of a tier, with optional recalibration on a small target sample.

In [ ]:
def cross_cohort(train_df, test_df, tier, recal_n=1000):
    cols = L.TIERS[tier]
    tr = train_df.dropna(subset=["time_y", "event"]); te = test_df.dropna(subset=cols)
    return L.run_models(specs(cols), tr, te, recal_n=recal_n, seed=SEED)

# DRY RUN on RICH sites only (a stand-in "cohort B"); no NAGALA data are used.
DRY_TEST_SITE = 8
dry_tr, dry_te = rich[rich.site != DRY_TEST_SITE], rich[rich.site == DRY_TEST_SITE]
dry_res, dry_fits = cross_cohort(dry_tr, dry_te, "T2", recal_n=500 if FAST else 1000)
dry_res[["model", "cindex", "scaled_brier", "oe", "cal_slope", "viol_rate"]].round(3)

## 5. Subgroups, calibration and decision curve (dry-run fit)

In [ ]:
f = dry_fits["CKINN"]
display_cols = ["n", "events", "cindex", "auc_3y", "scaled_brier", "oe", "cal_slope"]
test_g = dry_te.copy(); test_g["sex"] = np.where(test_g.sex_male == 1, "male", "female")
print(L.subgroup_metrics(f, test_g, "sex").round(3).to_string(index=False))
print(L.subgroup_metrics(f, test_g, "age_band").round(3).to_string(index=False))
dc = {n: L.decision_curve(fx, dry_te) for n, fx in dry_fits.items()}
fig, ax = plt.subplots(figsize=(5, 3.8))
for n, d in dc.items(): ax.plot(d.threshold * 100, d.net_benefit, label=n)
ax.plot(d.threshold * 100, d.treat_all, "k:", label="treat all"); ax.axhline(0, c="grey", lw=0.8)
ax.set_xlabel("threshold, %"); ax.set_ylabel("net benefit"); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(FIG / "04_decision_curve_dry.png"); plt.show()

## 6. Paired bootstrap: CKINN minus comparators (dry-run fit)

In [ ]:
NB = 20 if FAST else 500
for other in ["MLP", "XGB-Cox monotone"]:
    print(f"CKINN - {other}")
    print(L.boot_diff(dry_fits["CKINN"], dry_fits[other], dry_te, n_boot=NB, metrics=("cindex", "scaled_brier")).round(4))

## 7. R3 · Cross-cohort: RICH → NAGALA and NAGALA → RICH  (LOCKED)
Run only after the protocol is locked. Set `PROTOCOL_LOCKED = True` in the configuration cell, restart, and run all cells.

In [ ]:
def run_cross_cohort():
    rows = []
    for tier in ["T1", "T2", "T3", "T4"]:
        for src, dst, a, b in [("RICH", "NAGALA", rich, nag), ("NAGALA", "RICH", nag, rich)]:
            if tier == "T4" and src == "NAGALA":
                continue                  # T4 trained on complete NAGALA, tested on partly missing RICH: report separately
            res, _ = cross_cohort(a, b, tier)
            res.insert(0, "tier", tier); res.insert(1, "direction", f"{src} -> {dst}"); rows.append(res)
    out = pd.concat(rows, ignore_index=True); out.to_csv(OUT / "04_cross_cohort.csv", index=False)
    return out[["tier", "direction", "model", "cindex", "scaled_brier", "oe", "cal_slope", "viol_rate"]].round(3)

if PROTOCOL_LOCKED:
    cross = run_cross_cohort()
    display(cross)
else:
    print("LOCKED: cross-cohort analysis skipped. Lock protocol.md, fix lambda in the decision log, set PROTOCOL_LOCKED = True and rerun.")